In [3]:
import torch
import numpy as np
import time
import pickle

import hockey.hockey_env as h_env

from sac.plots import *
from sac.utils import from_dict, SB3_PARAMS, get_trained_agent, create_agent, save_logs
from sac.trainer import HockeyTrainer, SCORE_REWARD, DEFAULT_REWARD, ACTION_BOUNDS, SCALING
from sac.tournament import PooledTournament, LowMemTournament, MixedTournament

In [4]:
params = SB3_PARAMS
hidden_sizes = [256, 256]
params["alpha"] = 1.
params["lr"] = 3e-4
params["hl_bounds"] = [-30, 11]
params["hidden_sizes"] = hidden_sizes
params["action_bounds"] = ACTION_BOUNDS
params["obs_scale"] = SCALING
params["hl"] = True

In [38]:
agent1 = get_trained_agent(f"../agents/4000.pth", params)
agent2 = get_trained_agent(f"../agents/4000.pth", params)
trainer = HockeyTrainer(agent1, reward_func=SCORE_REWARD)

In [39]:
weak_opponent = h_env.BasicOpponent(weak=True)
strong_opponent = h_env.BasicOpponent(weak=False)

In [7]:
agent_list = [
    "s7000",
    "s5500",
    "current2",
    "500",
    "1800",
    "2300",
    "3100",
    "3400",
    "3700",
    "4000",
    "4400",
    "4902",
    "5612",
    "6212",
    "6716"
]

In [14]:
results = np.zeros((len(agent_list), len(agent_list)))

In [11]:
def update_agent(agent, name):
    agent_path = f"../agents/{name}.pth"
    state = torch.load(agent_path, map_location="cpu")
    agent.policy.restore_state(state[2])

In [15]:
for i, agent_name in enumerate(agent_list):
    for j in range(i+1, len(agent_list)):
        if results[i, j] == 0:
            agent2_name = agent_list[j]
            update_agent(agent1, agent_name)
            update_agent(agent2, agent2_name)
            rewards, scores = trainer.evaluate(opponent=agent2, episodes=500)
            winrate = 0.5* (np.mean(scores) + 1)
            results[i, j] = winrate
            results[j, i] = 1-winrate
            print(agent_name, "vs.", agent2_name, winrate)

s7000 vs. s5500 0.515
s7000 vs. current2 0.28500000000000003
s7000 vs. 500 0.7949999999999999
s7000 vs. 1800 0.525
s7000 vs. 2300 0.33499999999999996
s7000 vs. 3100 0.28800000000000003
s7000 vs. 3400 0.328
s7000 vs. 3700 0.198
s7000 vs. 4000 0.196
s7000 vs. 4400 0.21100000000000002
s7000 vs. 4902 0.325
s7000 vs. 5612 0.482
s7000 vs. 6212 0.349
s7000 vs. 6716 0.366
s5500 vs. current2 0.199
s5500 vs. 500 0.15899999999999997
s5500 vs. 1800 0.525
s5500 vs. 2300 0.187
s5500 vs. 3100 0.415
s5500 vs. 3400 0.497
s5500 vs. 3700 0.22899999999999998
s5500 vs. 4000 0.322
s5500 vs. 4400 0.502
s5500 vs. 4902 0.52
s5500 vs. 5612 0.484
s5500 vs. 6212 0.5
s5500 vs. 6716 0.554
current2 vs. 500 0.15500000000000003
current2 vs. 1800 0.492
current2 vs. 2300 0.06
current2 vs. 3100 0.422
current2 vs. 3400 0.5
current2 vs. 3700 0.416
current2 vs. 4000 0.476
current2 vs. 4400 0.449
current2 vs. 4902 0.476
current2 vs. 5612 0.464
current2 vs. 6212 0.471
current2 vs. 6716 0.446
500 vs. 1800 0.582
500 vs. 2300 0.

In [43]:
with open("res.npy", "wb") as outfile:
    np.save(outfile, all_res)

In [20]:
np.mean(results, axis=1)

array([0.34653333, 0.37186667, 0.42286667, 0.307     , 0.38426667,
       0.384     , 0.43806667, 0.42986667, 0.4076    , 0.45813333,
       0.58266667, 0.56493333, 0.61293333, 0.63186667, 0.6574    ])

In [23]:
np.min(results + np.eye(len(results)), axis=1)

array([0.196, 0.159, 0.06 , 0.167, 0.274, 0.07 , 0.097, 0.088, 0.008,
       0.015, 0.421, 0.252, 0.155, 0.5  , 0.446])

In [18]:
results

array([[0.   , 0.515, 0.285, 0.795, 0.525, 0.335, 0.288, 0.328, 0.198,
        0.196, 0.211, 0.325, 0.482, 0.349, 0.366],
       [0.485, 0.   , 0.199, 0.159, 0.525, 0.187, 0.415, 0.497, 0.229,
        0.322, 0.502, 0.52 , 0.484, 0.5  , 0.554],
       [0.715, 0.801, 0.   , 0.155, 0.492, 0.06 , 0.422, 0.5  , 0.416,
        0.476, 0.449, 0.476, 0.464, 0.471, 0.446],
       [0.205, 0.841, 0.845, 0.   , 0.582, 0.224, 0.195, 0.168, 0.213,
        0.211, 0.284, 0.261, 0.236, 0.167, 0.173],
       [0.475, 0.475, 0.508, 0.418, 0.   , 0.609, 0.437, 0.385, 0.355,
        0.337, 0.359, 0.412, 0.274, 0.373, 0.347],
       [0.665, 0.813, 0.94 , 0.776, 0.391, 0.   , 0.176, 0.178, 0.458,
        0.422, 0.329, 0.224, 0.212, 0.106, 0.07 ],
       [0.712, 0.585, 0.578, 0.805, 0.563, 0.824, 0.   , 0.305, 0.505,
        0.32 , 0.579, 0.282, 0.147, 0.269, 0.097],
       [0.672, 0.503, 0.5  , 0.832, 0.615, 0.822, 0.695, 0.   , 0.464,
        0.479, 0.246, 0.327, 0.116, 0.089, 0.088],
       [0.802, 0.771, 0.

In [24]:
update_agent(trainer.agent, "6716")

In [25]:
rewards, scores = trainer.evaluate(opponent=weak_opponent, episodes=500)
winrate = 0.5* (np.mean(scores) + 1)
winrate

np.float64(0.874)

In [26]:
rewards, scores = trainer.evaluate(opponent=strong_opponent, episodes=500)
winrate = 0.5* (np.mean(scores) + 1)
winrate

np.float64(0.931)

In [27]:
bot_results = np.zeros((len(agent_list),2))

In [40]:
for i, agent_name in enumerate(agent_list):
        update_agent(agent1, agent_name)
        rewards, scores = trainer.evaluate(opponent=weak_opponent, episodes=500)
        winrate = 0.5* (np.mean(scores) + 1)
        bot_results[i, 0] = winrate
        print(agent_name, "vs. Weak:", winrate)
        rewards, scores = trainer.evaluate(opponent=strong_opponent, episodes=500)
        winrate = 0.5* (np.mean(scores) + 1)
        bot_results[i, 1] = winrate
        print(agent_name, "vs. Strong:", winrate)

s7000 vs. Weak: 0.743
s7000 vs. Strong: 0.774
s5500 vs. Weak: 0.862
s5500 vs. Strong: 0.8140000000000001
current2 vs. Weak: 0.917
current2 vs. Strong: 0.89
500 vs. Weak: 0.952
500 vs. Strong: 0.638
1800 vs. Weak: 0.724
1800 vs. Strong: 0.646
2300 vs. Weak: 0.863
2300 vs. Strong: 0.8009999999999999
3100 vs. Weak: 0.881
3100 vs. Strong: 0.889
3400 vs. Weak: 0.958
3400 vs. Strong: 0.758
3700 vs. Weak: 0.911
3700 vs. Strong: 0.9359999999999999
4000 vs. Weak: 0.8029999999999999
4000 vs. Strong: 0.94
4400 vs. Weak: 0.759
4400 vs. Strong: 0.901
4902 vs. Weak: 0.9319999999999999
4902 vs. Strong: 0.837
5612 vs. Weak: 0.852
5612 vs. Strong: 0.9510000000000001
6212 vs. Weak: 0.9079999999999999
6212 vs. Strong: 0.9670000000000001
6716 vs. Weak: 0.869
6716 vs. Strong: 0.935


In [33]:
class WeakWrapper:

    def __init__(self):
        pass

    def act(self, o, noise_scale=None):
        return weak_opponent.act(o)

In [34]:
trainer.agent = WeakWrapper()

rewards, scores = trainer.evaluate(opponent=strong_opponent, episodes=500)
winrate = 0.5* (np.mean(scores) + 1)
winrate

np.float64(0.40700000000000003)

In [41]:
k = len(agent_list)
all_res = np.zeros((k+2, k+2))
all_res[2:, 2:] = results
all_res[2:, :2] = bot_results
all_res[:2, 2:] = bot_results.T
all_res[0, 1] = winrate
all_res[1, 0] = 1 - winrate

In [42]:
all_res

array([[0.   , 0.935, 0.743, 0.862, 0.917, 0.952, 0.724, 0.863, 0.881,
        0.958, 0.911, 0.803, 0.759, 0.932, 0.852, 0.908, 0.869],
       [0.065, 0.   , 0.774, 0.814, 0.89 , 0.638, 0.646, 0.801, 0.889,
        0.758, 0.936, 0.94 , 0.901, 0.837, 0.951, 0.967, 0.935],
       [0.743, 0.774, 0.   , 0.515, 0.285, 0.795, 0.525, 0.335, 0.288,
        0.328, 0.198, 0.196, 0.211, 0.325, 0.482, 0.349, 0.366],
       [0.862, 0.814, 0.485, 0.   , 0.199, 0.159, 0.525, 0.187, 0.415,
        0.497, 0.229, 0.322, 0.502, 0.52 , 0.484, 0.5  , 0.554],
       [0.917, 0.89 , 0.715, 0.801, 0.   , 0.155, 0.492, 0.06 , 0.422,
        0.5  , 0.416, 0.476, 0.449, 0.476, 0.464, 0.471, 0.446],
       [0.952, 0.638, 0.205, 0.841, 0.845, 0.   , 0.582, 0.224, 0.195,
        0.168, 0.213, 0.211, 0.284, 0.261, 0.236, 0.167, 0.173],
       [0.724, 0.646, 0.475, 0.475, 0.508, 0.418, 0.   , 0.609, 0.437,
        0.385, 0.355, 0.337, 0.359, 0.412, 0.274, 0.373, 0.347],
       [0.863, 0.801, 0.665, 0.813, 0.94 , 0.776